# Step 7 -- Pixel-based Anomaly Segmentation Baselines (ERFNet)

**Objective:** Run post-hoc anomaly detection methods on a **pixel-based**
semantic segmentation model (ERFNet) across five anomaly validation datasets.

**Methods implemented:**
- **MSP** (Maximum Softmax Probability): $1 - \max_c P(y=c|x)$
- **Max Logit**: $-\max_c f_c(x)$
- **Max Entropy**: $H(P) = -\sum_c P(y=c|x) \log P(y=c|x)$

**Datasets:** SMIYC RoadAnomaly21, SMIYC RoadObsticle21, FS Lost&Found,
FS Static, Road Anomaly

**Metrics:** AuPRC (area under precision-recall curve),
FPR@95TPR (false positive rate at 95% true positive rate)

## 1. Imports & Environment Setup

In [ ]:
import os
import glob
import json
import random
import os.path as osp

import torch
import numpy as np
from PIL import Image
from torch import nn, optim
from torchvision.datasets import Cityscapes
from torchvision.transforms import Compose, Resize, ToTensor

# --- Reproducibility ---
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

# --- ERFNet constants ---
NUM_CHANNELS = 3
NUM_CLASSES = 20
IGNORE_INDEX = 19

# --- Transforms (same as the baseline) ---
input_transform = Compose([
    Resize((512, 1024), Image.BILINEAR),
    ToTensor(),
])

target_transform = Compose([
    Resize((512, 1024), Image.NEAREST),
])

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 2. Evaluation Metrics (pure NumPy)

We implement AuPRC and FPR@95TPR **from scratch** without relying on
`sklearn` or external libraries, making the notebook self-contained.

In [ ]:
def average_precision_score(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """Area under the precision-recall curve (no sklearn dependency)."""
    y_true = np.asarray(y_true).astype(np.int64)
    y_score = np.asarray(y_score).astype(np.float64)
    if y_true.ndim != 1 or y_score.ndim != 1 or y_true.shape[0] != y_score.shape[0]:
        raise ValueError("y_true and y_score must be 1D arrays of the same length")

    pos = int(np.sum(y_true == 1))
    if pos == 0:
        return 0.0

    order = np.argsort(-y_score, kind="mergesort")
    y_true_sorted = y_true[order]

    tp = np.cumsum(y_true_sorted == 1)
    fp = np.cumsum(y_true_sorted == 0)

    precision = tp / np.maximum(tp + fp, 1)
    recall = tp / pos

    # Only keep distinct score thresholds
    distinct = np.r_[True, y_score[order][1:] != y_score[order][:-1]]
    precision = precision[distinct]
    recall = recall[distinct]

    recall = np.r_[0.0, recall]
    precision = np.r_[precision[0], precision]

    return float(np.sum((recall[1:] - recall[:-1]) * precision[1:]))


def fpr_at_95_tpr(y_score: np.ndarray, y_true: np.ndarray) -> float:
    """False positive rate when true positive rate reaches 95%."""
    y_true = np.asarray(y_true).astype(np.int64)
    y_score = np.asarray(y_score).astype(np.float64)
    if y_true.ndim != 1 or y_score.ndim != 1 or y_true.shape[0] != y_score.shape[0]:
        raise ValueError("y_true and y_score must be 1D arrays of the same length")

    pos = int(np.sum(y_true == 1))
    neg = int(np.sum(y_true == 0))
    if pos == 0 or neg == 0:
        return 0.0

    order = np.argsort(-y_score, kind="mergesort")
    y_true_sorted = y_true[order]

    tp = np.cumsum(y_true_sorted == 1)
    fp = np.cumsum(y_true_sorted == 0)

    tpr = tp / pos
    fpr = fp / neg

    idx = np.where(tpr >= 0.95)[0]
    if idx.size == 0:
        return 1.0
    return float(np.min(fpr[idx]))


print("Metrics defined: average_precision_score, fpr_at_95_tpr")


## Mean IoU (Cityscapes)

`calculate_miou` builds a confusion matrix from flat prediction/ground-truth
arrays (ignoring `ignore_index` pixels) and averages per-class IoU over all
classes present in the ground truth.

In [ ]:
def calculate_miou(predictions, ground_truth, num_classes, ignore_index=255):
    preds = predictions.flatten()
    gts = ground_truth.flatten()
    valid_mask = (gts != ignore_index)
    preds = preds[valid_mask]
    gts = gts[valid_mask]
    hist = np.bincount(num_classes * gts + preds,
                       minlength=num_classes ** 2).reshape(num_classes, num_classes)
    intersection = np.diag(hist)
    ground_truth_set = hist.sum(axis=1)
    predicted_set = hist.sum(axis=0)
    union = ground_truth_set + predicted_set - intersection
    valid_classes = union > 0
    intersection = intersection[valid_classes]
    union = union[valid_classes]
    iou = intersection / union
    miou = np.mean(iou)
    return miou, iou

## 3. Anomaly Score Functions

Given per-pixel logits $f \in \mathbb{R}^{C \times H \times W}$,
each method produces an **anomaly heatmap** $A \in \mathbb{R}^{H \times W}$
where higher values indicate more anomalous pixels.

| Method | Formula | Intuition |
|--------|---------|-----------|
| **MSP** | $1 - \max_c \text{softmax}(f/T)_c$ | Low confidence -> anomaly |
| **Max Logit** | $-\max_c f_c$ | Low raw activation -> anomaly |
| **Max Entropy** | $-\sum_c p_c \log p_c$ | Uniform prediction -> anomaly |

Temperature $T$ scales logits before softmax: $p = \text{softmax}(f/T)$.
$T > 1$ flattens the distribution (more conservative).

In [ ]:
def anomaly_score_from_logits(
    logits: torch.Tensor, method: str, temperature: float = 1.0
) -> np.ndarray:
    """Convert per-pixel logits to an anomaly score map.

    Args:
        logits: Tensor of shape (B, C, H, W) -- raw model output.
        method: One of {"msp", "max_logit", "max_entropy"}.
        temperature: T > 0 for temperature scaling (default 1.0 = no scaling).

    Returns:
        np.ndarray of shape (H, W) -- anomaly scores.
    """
    temperature = max(float(temperature), 1e-8)
    scaled_logits = logits / temperature

    if method == "msp":
        probs = torch.softmax(scaled_logits, dim=1)
        score = 1.0 - probs.max(dim=1).values
        return score.squeeze(0).detach().cpu().numpy()

    if method == "max_logit":
        score = -logits.max(dim=1).values
        return score.squeeze(0).detach().cpu().numpy()

    if method == "max_entropy":
        log_probs = torch.log_softmax(scaled_logits, dim=1)
        probs = log_probs.exp()
        entropy = -(probs * log_probs).sum(dim=1)
        return entropy.squeeze(0).detach().cpu().numpy()

    raise ValueError(f"Unknown method: {method}")


print("anomaly_score_from_logits ready (msp | max_logit | max_entropy)")


## 4. Dataset Helpers

Each anomaly dataset has a slightly different ground-truth format:

- **RoadAnomaly / RoadAnomaly21**: label `2` -> anomaly
- **FS Lost&Found**: label `0` = void, `1` = road, `>1` -> anomaly
- **RoadObsticle21 / fs_static**: standard binary (0 = in-dist, 1 = anomaly)

In [ ]:
def infer_dataset_name(input_pattern: str) -> str:
    """Extract dataset name from input glob pattern."""
    norm = input_pattern.replace("\\", "/")
    parts = [p for p in norm.split("/") if p]
    if "Validation_Dataset" in parts:
        idx = parts.index("Validation_Dataset")
        if idx + 1 < len(parts):
            return parts[idx + 1]
    if len(parts) >= 2:
        return parts[-2]
    return input_pattern


def load_gt_mask(path: str, pred_size: tuple) -> np.ndarray | None:
    """Load and normalise a ground-truth anomaly mask.

    Returns None if the mask file is missing.
    """
    pathGT = path.replace("images", "labels_masks")
    # Fix extension mismatches in some datasets
    if "RoadObsticle21" in pathGT:
        pathGT = pathGT.replace("webp", "png")
    if "fs_static" in pathGT:
        pathGT = pathGT.replace("jpg", "png")
    if "RoadAnomaly" in pathGT:
        pathGT = pathGT.replace("jpg", "png")

    if not osp.exists(pathGT):
        return None

    mask = Image.open(pathGT)
    if mask.size != (pred_size[1], pred_size[0]):
        mask = mask.resize((pred_size[1], pred_size[0]), Image.NEAREST)
    ood_gts = np.array(mask)

    # --- Per-dataset normalisation to {0: in-dist, 1: anomaly} ---
    if "RoadAnomaly" in pathGT:                      # id 2 = anomaly
        ood_gts = np.where((ood_gts == 2), 1, ood_gts)

    if ("LostAndFound" in pathGT) or ("LostFound" in pathGT) or ("FS_LostFound_full" in pathGT):
        unique_vals = set(np.unique(ood_gts).tolist())
        if not unique_vals.issubset({0, 1, 255}):
            ood_gts = np.where((ood_gts == 0), 255, ood_gts)
            ood_gts = np.where((ood_gts == 1), 0, ood_gts)
            ood_gts = np.where((ood_gts > 1) & (ood_gts < 201), 1, ood_gts)

    if "Streethazard" in pathGT:
        ood_gts = np.where((ood_gts == 14), 255, ood_gts)
        ood_gts = np.where((ood_gts < 20), 0, ood_gts)
        ood_gts = np.where((ood_gts == 255), 1, ood_gts)

    return ood_gts


print("Dataset helpers ready")


## 5. Load ERFNet Model

In [ ]:
from erfnet import ERFNet

MODEL_DIR = "../trained_models/"
MODEL_WEIGHTS = "erfnet_pretrained.pth"

print(f"Loading model from: {MODEL_DIR}")
print(f"Weights: {MODEL_WEIGHTS}")

model = ERFNet(NUM_CLASSES)

if device.type == "cuda":
    model = torch.nn.DataParallel(model).to(device)
else:
    model = model.to(device)


def load_my_state_dict(model, state_dict):
    """Custom loader -- handles 'module.' prefix from DataParallel."""
    own_state = model.state_dict()
    for name, param in state_dict.items():
        if name not in own_state:
            if name.startswith("module."):
                own_state[name.split("module.")[-1]].copy_(param)
            else:
                print(f"  [skip] {name}")
                continue
        else:
            own_state[name].copy_(param)
    return model


weightspath = osp.join(MODEL_DIR, MODEL_WEIGHTS)
state = torch.load(weightspath, map_location=lambda storage, loc: storage)
model = load_my_state_dict(model, state)
model.eval()

print("ERFNet model loaded and ready for inference.")


## 6. Run Evaluation -- All Datasets x All Methods

Loop through every validation dataset, run the forward pass once,
compute anomaly scores for each method, and record AuPRC / FPR@95TPR.

In [ ]:
from collections import defaultdict

# --- Dataset paths (relative to this notebook's location in eval/) ---
DATASETS = {
    "RoadAnomaly21":  "../Validation_Dataset/RoadAnomaly21/images/*.png",
    "RoadObsticle21": "../Validation_Dataset/RoadObsticle21/images/*.webp",
    "FS_LostFound":   "../Validation_Dataset/FS_LostFound_full/images/*.png",
    "FS_Static":      "../Validation_Dataset/fs_static/images/*.jpg",
    "RoadAnomaly":    "../Validation_Dataset/RoadAnomaly/images/*.jpg",
}

METHODS = ["msp", "max_logit", "max_entropy"]

# results[dataset_name][method] = {"auprc": ..., "fpr95": ...}
results = {}

for ds_name, ds_pattern in DATASETS.items():
    results[ds_name] = {}
    image_paths = sorted(glob.glob(os.path.expanduser(ds_pattern)))
    print(f"\n{'='*60}")
    print(f"Dataset: {ds_name}  ({len(image_paths)} images)")
    print(f"{'='*60}")

    if len(image_paths) == 0:
        print("  WARNING: No images found -- skipping.")
        continue

    # --- Collect all logits + ground truths (one forward pass per image) ---
    all_logits = []      # list of (C, H, W) tensors on CPU
    all_gts = []         # list of (H, W) np arrays

    for path in image_paths:
        img = Image.open(path).convert("RGB")
        img_tensor = input_transform(img).unsqueeze(0).float().to(device)

        with torch.no_grad():
            logits = model(img_tensor)          # (1, C, H, W)

        gt = load_gt_mask(path, pred_size=logits.shape[-2:])
        if gt is None:
            continue
        if 1 not in np.unique(gt):
            continue

        all_logits.append(logits.squeeze(0).cpu())
        all_gts.append(gt)
        torch.cuda.empty_cache()

    if len(all_gts) == 0:
        print("  No valid samples with anomaly pixels.")
        continue

    # --- Evaluate each method on the SAME logits ---
    for method in METHODS:
        anomaly_scores = []
        for logits_tensor in all_logits:
            score_map = anomaly_score_from_logits(
                logits_tensor.unsqueeze(0), method, temperature=1.0
            )
            anomaly_scores.append(score_map)

        # Flatten all pixels across the dataset
        gt_all = np.concatenate([g.flatten() for g in all_gts])
        scores_all = np.concatenate([s.flatten() for s in anomaly_scores])

        ood_mask = (gt_all == 1)
        ind_mask = (gt_all == 0)

        ood_out = scores_all[ood_mask]
        ind_out = scores_all[ind_mask]

        val_out = np.concatenate([ind_out, ood_out])
        val_label = np.concatenate([np.zeros(len(ind_out)), np.ones(len(ood_out))])

        auprc = average_precision_score(val_label, val_out)
        fpr95 = fpr_at_95_tpr(val_out, val_label)

        results[ds_name][method] = {"auprc": auprc, "fpr95": fpr95}
        print(f"  {method:>12s}  |  AuPRC: {auprc*100:5.2f}%  |  FPR@95: {fpr95*100:5.2f}%")

print("\n\nEvaluation complete!")


## 7. Temperature Scaling

Fit a single temperature $T$ on the **Cityscapes validation set** by
minimising the negative log-likelihood (NLL), then re-evaluate MSP at
different temperatures.

> **Pro tip:** Save the model logits to disk first, then try different
> $T$ values without re-running the forward pass.

In [ ]:
# --- 7a. Define temperature scaler & Cityscapes helpers ---

from dataset import cityscapes
from temperature_scaling import _ECELoss

print("Collecting logits on Cityscapes validation set for temperature fitting...")

# Label mapping (labelId -> trainId)
class LabelIdsToTrainIds:
    def __init__(self, ignore_index: int = IGNORE_INDEX):
        mapping = np.full(256, 255, dtype=np.uint8)
        for cls in Cityscapes.classes:
            if cls.id < 0:
                continue
            train_id = cls.train_id
            if train_id == 255 or cls.ignore_in_eval:
                mapping[cls.id] = 255
            else:
                mapping[cls.id] = train_id
        mapping[255] = 255
        self.mapping = mapping
        self.ignore_index = ignore_index

    def __call__(self, image):
        label_ids = np.array(image, dtype=np.uint8)
        train_ids = self.mapping[label_ids]
        train_ids[train_ids == 255] = self.ignore_index
        return torch.from_numpy(train_ids.astype(np.int64)).unsqueeze(0)


input_transform_cs = Compose([Resize(512, Image.BILINEAR), ToTensor()])
target_transform_cs = Compose([Resize(512, Image.NEAREST), LabelIdsToTrainIds()])


class SegmentationTemperatureScaler(nn.Module):
    """Learnable temperature parameter for logit calibration."""

    def __init__(self, init_temperature: float = 1.5):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * float(init_temperature))

    def temperature_scale(self, logits: torch.Tensor) -> torch.Tensor:
        return logits / self.temperature.clamp_min(1e-8)

    def set_temperature(self, logits, labels, device):
        nll_criterion = nn.CrossEntropyLoss().to(device)
        ece_criterion = _ECELoss().to(device)

        logits = logits.to(device)
        labels = labels.to(device)

        before_nll = nll_criterion(logits, labels).item()
        before_ece = ece_criterion(logits, labels).item()
        print(f"Before temperature - NLL: {before_nll:.4f}, ECE: {before_ece:.4f}")

        optimizer = optim.LBFGS([self.temperature], lr=0.01, max_iter=200)

        def closure():
            optimizer.zero_grad()
            loss = nll_criterion(self.temperature_scale(logits), labels)
            loss.backward()
            return loss

        optimizer.step(closure)

        after_nll = nll_criterion(self.temperature_scale(logits), labels).item()
        after_ece = ece_criterion(self.temperature_scale(logits), labels).item()
        best_t = float(self.temperature.item())

        print(f"Optimal temperature: {best_t:.4f}")
        print(f"After temperature  - NLL: {after_nll:.4f}, ECE: {after_ece:.4f}")

        return {
            "temperature": best_t,
            "before_nll": before_nll, "after_nll": after_nll,
            "before_ece": before_ece, "after_ece": after_ece,
        }


In [ ]:
# --- 7b. Collect Cityscapes validation logits & fit T ---

CITYSCAPES_DIR = "../Cityscapes val"   # <-- adjust to your Cityscapes path
ERFNET_MIOU = None

if not osp.exists(CITYSCAPES_DIR):
    print(f"WARNING: Cityscapes directory not found at '{CITYSCAPES_DIR}'.")
    print("Temperature scaling requires the Cityscapes validation set.")
    print("Please download it and update CITYSCAPES_DIR.")
else:
    loader = torch.utils.data.DataLoader(
        cityscapes(CITYSCAPES_DIR, input_transform_cs, target_transform_cs,
                   subset="val", label_suffix="_labelIds.png"),
        num_workers=4, batch_size=1, shuffle=False,
    )

    logits_list, labels_list = [], []
    MAX_PIXELS_PER_IMAGE = 4096
    conf_matrix = np.zeros((20, 20), dtype=np.int64)

    with torch.no_grad():
        for step, (images, labels, filename, _) in enumerate(loader):
            images = images.to(device)
            outputs = model(images)

            # Full-resolution argmax for mIoU (before any subsampling)
            pred_full = outputs.argmax(dim=1).squeeze(0).cpu().numpy().astype(np.int64)
            gt_full = labels.squeeze(0).squeeze(0).cpu().numpy().astype(np.int64)
            valid_px = gt_full != IGNORE_INDEX
            p_flat = pred_full[valid_px].ravel()
            g_flat = gt_full[valid_px].ravel()
            conf_matrix += np.bincount(20 * g_flat + p_flat, minlength=400).reshape(20, 20)

            # Flatten & filter ignore pixels
            logits_flat = outputs.permute(0, 2, 3, 1).reshape(-1, outputs.shape[1])
            labels_flat = labels.squeeze(1).reshape(-1)
            valid = labels_flat != IGNORE_INDEX
            logits_flat = logits_flat[valid]
            labels_flat = labels_flat[valid]

            if logits_flat.numel() == 0:
                continue

            # Subsample to limit memory
            if logits_flat.shape[0] > MAX_PIXELS_PER_IMAGE:
                idx = torch.randperm(logits_flat.shape[0])[:MAX_PIXELS_PER_IMAGE]
                logits_flat = logits_flat[idx]
                labels_flat = labels_flat[idx]

            logits_list.append(logits_flat.cpu())
            labels_list.append(labels_flat.cpu())

            if step % 25 == 0:
                print(f"  [{step}] {logits_flat.shape[0]} valid pixels "
                      f"from {osp.basename(filename[0])}")

    # mIoU over classes 0..18; class 19 (void) excluded from the average.
    # GT void pixels are already filtered (IGNORE_INDEX=19); void predictions
    # land in column 19 and correctly increase FN for the GT class they miss.
    intersection   = np.diag(conf_matrix[:19, :19])
    gt_per_class   = conf_matrix[:19, :].sum(axis=1)
    pred_per_class = conf_matrix[:19, :19].sum(axis=0)
    union          = gt_per_class + pred_per_class - intersection
    valid_cls      = union > 0
    iou_per_class  = np.where(valid_cls,
                              intersection / np.where(valid_cls, union, 1.0),
                              0.0)
    ERFNET_MIOU = float(np.mean(iou_per_class[valid_cls])) * 100
    print(f"\nERFNet mIoU on Cityscapes val (19 classes): {ERFNET_MIOU:.2f}%")

    logits_all = torch.cat(logits_list, dim=0)
    labels_all = torch.cat(labels_list, dim=0)
    print(f"Collected logits: {tuple(logits_all.shape)}, "
          f"labels: {tuple(labels_all.shape)}")

    # --- Fit temperature ---
    scaler = SegmentationTemperatureScaler(init_temperature=1.5).to(device)
    stats = scaler.set_temperature(logits_all, labels_all, device)
    BEST_T_ERFNET = stats["temperature"]
    print(f"\nBest temperature for ERFNet: {BEST_T_ERFNET:.4f}")

## 8. Results Summary Table

Compile all results into the table format required by the assignment.

| Model | mIoU | Method | SMIYC RA-21 AuPRC | SMIYC RA-21 FPR95 | ... | Road Anomaly AuPRC | Road Anomaly FPR95 |
|-------|------|--------|-------------------|--------------------|-----|---------------------|---------------------|
| ERFNet | -- | MSP | ... | ... | ... | ... | ... |
| ERFNet | -- | Max Logit | ... | ... | ... | ... | ... |
| ERFNet | -- | Max Entropy | ... | ... | ... | ... | ... |


In [ ]:
def build_results_table(results_dict, model_name="ERFNet", miou="--"):
    """Print results in a flat format."""
    print(f"\n{'='*80}")
    print(f"Model: {model_name}  |  mIoU: {miou}")
    print(f"{'='*80}")
    print(f"{'Method':>12s} | {'Dataset':<20s} | {'AuPRC':>8s} | {'FPR@95':>8s}")
    print(f"{'-'*12}-+-{'-'*20}-+-{'-'*8}-+-{'-'*8}")

    for ds_name in DATASETS:
        if ds_name not in results_dict:
            continue
        for method in METHODS:
            if method not in results_dict[ds_name]:
                continue
            r = results_dict[ds_name][method]
            print(f"{method:>12s} | {ds_name:<20s} "
                  f"| {r['auprc']*100:7.2f}% | {r['fpr95']*100:7.2f}%")


miou_str = f"{ERFNET_MIOU:.2f}%" if ERFNET_MIOU is not None else "--"
build_results_table(results, model_name="ERFNet", miou=miou_str)

# --- Optional: pivot table with pandas ---
try:
    import pandas as pd
    rows = []
    for ds_name in results:
        for method in results[ds_name]:
            r = results[ds_name][method]
            rows.append({
                "Model": "ERFNet", "mIoU": miou_str, "Method": method,
                "Dataset": ds_name,
                "AuPRC": round(r["auprc"] * 100, 2),
                "FPR95": round(r["fpr95"] * 100, 2),
            })
    df = pd.DataFrame(rows)
    pivot = df.pivot_table(
        index=["Model", "mIoU", "Method"],
        columns="Dataset",
        values=["AuPRC", "FPR95"],
        aggfunc="first"
    )
    print("\n\n=== Assignment-ready pivot table ===")
    print(pivot.to_string())
except ImportError:
    print("\n(pandas not available -- install 'pip install pandas' for pivot table)")

**End of Step 7 notebook.**